# 💳 Project Task: GoPay Fintech Analytics
## Data Cleaning, Feature Engineering & Exploratory Data Analysis

**Module 2 — Python for Data Analysis | Purwadhika Digital Technology School**

---

> ⚠️ **Jangan di-run dulu.** Copy notebook ini terlebih dahulu, baru kerjakan di file copy-an kamu.

---

### Konteks Bisnis
GoPay telah berkembang menjadi tulang punggung ekosistem Super App GoTo. Fitur GoPayLater — layanan kredit berbasis limit — berhasil mendongkrak GTV, namun kini menghadapi masalah serius: tingkat NPL (Non-Performing Loan) yang meningkat, bug validasi limit kredit, dan inkonsistensi data dari puluhan micro-service.

Kamu berperan sebagai Data Analyst di tim **Risk Management GoPay** yang diminta untuk membersihkan data, mengidentifikasi pola gagal bayar, dan memberikan rekomendasi perbaikan credit scoring.

**Dataset (3 tabel):**
- `gopay_users.csv` — 35.000 baris (Dimensi User)
- `gopay_services.csv` — 20 baris (Dimensi Layanan)
- `gopay_transactions.csv` — 300.000 baris (Fakta Transaksi)

---

### ⚠️ Catatan Penting
- Task ini **open-ended** — tidak ada satu jawaban yang mutlak benar
- Yang dinilai: **ketepatan keputusan**, **kualitas justifikasi**, dan **kedalaman analisis**
- Setiap keputusan di Data Cleaning & Feature Engineering **wajib disertai penjelasan** di markdown cell
- EDA dikerjakan **tanpa visualisasi** — gunakan pandas aggregation, filtering, sorting, dan merge

---

### 🚨 Business Context Error (Wajib Diinvestigasi)
> Lebih dari **50% transaksi GoPayLater** memiliki `amount` yang **melebihi `paylater_limit`** user yang bersangkutan.  
> Ini adalah bug sistematis pada validasi limit kredit — bukan sekadar outlier biasa.  
> Identifikasi, kuantifikasi dampak finansialnya, dan rekomendasikan perbaikan.

---
## 0. Import & Load Data

In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
# Load semua dataset
# Sesuaikan path dengan lokasi file kamu
df_users_raw    = pd.read_csv('dataset/gopay_users.csv')
df_services_raw = pd.read_csv('dataset/gopay_services.csv')
df_trx_raw      = pd.read_csv('dataset/gopay_transactions.csv')

# Buat copy untuk dikerjakan
df_users    = df_users_raw.copy()
df_services = df_services_raw.copy()
df_trx      = df_trx_raw.copy()

print(f'users       : {df_users.shape}')
print(f'services    : {df_services.shape}')
print(f'transactions: {df_trx.shape}')

users       : (35000, 5)
services    : (20, 3)
transactions: (300000, 8)


---
## 2. Data Cleaning

### 2.1 Eksplorasi Awal (Wajib)

Lakukan eksplorasi menyeluruh pada **ketiga tabel** sebelum membersihkan data apapun.

In [4]:
# Shape dan info umum — lakukan untuk ketiga tabel
print("Shape dan info umum dataset df_users.csv")
df_users.info()
print("Shape dataset df_users.csv:", df_users.shape)

print("\nShape dan info umum dataset df_services.csv")
df_services.info()
print("Shape dataset df_services.csv:", df_services.shape)

print("\nShape dan info umum dataset df_trx.csv")
df_trx_raw.info()
print("Shape dataset df_trx.csv:", df_trx.shape)

Shape dan info umum dataset df_users.csv
<class 'pandas.DataFrame'>
RangeIndex: 35000 entries, 0 to 34999
Data columns (total 5 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                35000 non-null  str    
 1   join_date              35000 non-null  str    
 2   gopay_tier             35000 non-null  str    
 3   internal_credit_score  29750 non-null  float64
 4   paylater_limit         35000 non-null  int64  
dtypes: float64(1), int64(1), str(3)
memory usage: 2.1 MB
Shape dataset df_users.csv: (35000, 5)

Shape dan info umum dataset df_services.csv
<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   service_id    20 non-null     str  
 1   service_name  20 non-null     str  
 2   category      20 non-null     str  
dtypes: str(3)
memory usage: 1.2 KB
Shape dataset df_service

In [5]:
# Tipe data seluruh kolom
print(f"Tipe data tabel Users:\n{df_users.dtypes}\n", )
print(f"Tipe data tabel Services:\n{df_services.dtypes}\n", )
print(f"Tipe data tabel Transactions:\n{df_trx.dtypes}\n", )

Tipe data tabel Users:
user_id                      str
join_date                    str
gopay_tier                   str
internal_credit_score    float64
paylater_limit             int64
dtype: object

Tipe data tabel Services:
service_id      str
service_name    str
category        str
dtype: object

Tipe data tabel Transactions:
trx_id                str
user_id               str
service_id            str
trx_date              str
payment_method        str
amount              int64
late_fee          float64
payment_status        str
dtype: object



In [6]:
# Missing values: jumlah dan persentase per kolom, per tabel
# Tampilkan hanya kolom yang memiliki missing values

def missing(df):
    na_val = df.isnull().sum()
    na_val = na_val.apply(lambda x: f"{x} - {x/len(df):.2%}")
    return na_val

print("Missing Values pada tabel Users: jumlah dan persentase per kolom, per tabel")
display(missing(df_users))

print("internal_credit_score missing:")
df_users_na = df_users[df_users['internal_credit_score'].isna()]
display(df_users_na)

# print("Cek apakah di 'internal_credit_score' ada tipe data selain integer")
# naval_users_string = df_users["internal_credit_score"].apply(lambda x: isinstance(x, str)).sum()
# display(naval_users_string)

# df_services
print("Missing Values pada tabel Services: jumlah dan persentase per kolom, per tabel")
display(missing(df_services))

# df_trx
print("Missing Values pada tabel Transactions: jumlah dan persentase per kolom, per tabel")
display(missing(df_trx))

Missing Values pada tabel Users: jumlah dan persentase per kolom, per tabel


user_id                      0 - 0.00%
join_date                    0 - 0.00%
gopay_tier                   0 - 0.00%
internal_credit_score    5250 - 15.00%
paylater_limit               0 - 0.00%
dtype: str

internal_credit_score missing:


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit
0,GP-000001,2022-05-26,Basic,NaN,0
7,GP-000008,2023-03-26,Plus,NaN,500000
9,GP-000010,2021-04-20,Plus,NaN,5000000
19,GP-000020,2022-06-23,Plus,NaN,3000000
25,GP-000026,2022-01-15,Plus,NaN,500000
...,...,...,...,...,...
34977,GP-034978,2022-04-26,Plus,NaN,500000
34980,GP-034981,2023-02-28,Plus,NaN,1500000
34988,GP-034989,2021-03-16,Plus,NaN,1500000
34991,GP-034992,2022-10-19,Basic,NaN,0


Missing Values pada tabel Services: jumlah dan persentase per kolom, per tabel


service_id      0 - 0.00%
service_name    0 - 0.00%
category        0 - 0.00%
dtype: str

Missing Values pada tabel Transactions: jumlah dan persentase per kolom, per tabel


trx_id            0 - 0.00%
user_id           0 - 0.00%
service_id        0 - 0.00%
trx_date          0 - 0.00%
payment_method    0 - 0.00%
amount            0 - 0.00%
late_fee          0 - 0.00%
payment_status    0 - 0.00%
dtype: str

In [28]:
# Distribusi kolom-kolom kritis
# payment_method, amount, late_fee, payment_status, gopay_tier, internal_credit_score
print("Merge data users dan transactions")
dfMerge_tu = pd.merge(df_users, df_trx, on="user_id", how="left")
display(dfMerge_tu)

dfMerge_des = dfMerge_tu[["amount", "late_fee", "internal_credit_score"]].describe()
display(dfMerge_des)

dfMerge_vc = dfMerge_tu[["payment_method", "payment_status", "gopay_tier"]].value_counts()
display(dfMerge_vc)

Merge data users dan transactions


,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,GoPay
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,GoPay
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,PayLater
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,GoPay


,amount,late_fee,internal_credit_score
count,300000.00,300000.00,254714.00
mean,447504.65,26734.99,534.66
std,490747.56,1591437.75,153.57
min,15004.00,-15000.00,300.00
25%,213294.75,0.00,415.00
50%,413388.50,0.00,490.00
75%,612431.75,0.00,662.00
max,9965039.00,99999999.00,849.00


payment_method  payment_status  gopay_tier
GoPay           Paid            Plus          54144
gopay           Paid            Plus          36298
GoPay           Paid            Basic         36096
GoPayLater      Paid            Plus          27091
gopay           Paid            Basic         23556
GoPayLater      Paid            Basic         18038
PayLater        Paid            Plus          13679
CASH            Paid            Plus           9116
GO-PAY          Paid            Plus           9060
Cash            Paid            Plus           9045
PayLater        Paid            Basic          8852
gopay_later     Paid            Plus           6656
GO-PAY          Paid            Basic          6076
Cash            Paid            Basic          5933
CASH            Paid            Basic          5856
GoPayLater      Default         Plus           5412
gopay_later     Paid            Basic          4425
GoPayLater      Pending         Plus           3635
                Defau

In [29]:
# Cek konsistensi relasi antar tabel

# Apakah semua service_id di transactions ada di services?
trx_u = df_trx["service_id"].unique()
srv_u = df_services["service_id"].unique()

comparison_ts = df_trx["service_id"].isin(df_services["service_id"].unique())
display(comparison_ts.unique())
# Ya, semua service_id transaction ada di services

# Apakah semua user_id di transactions ada di users?
user_compare = df_trx["user_id"].isin(df_users["user_id"].unique())
display(user_compare.unique())
# Ya, semua user_id ada di transactions


array([ True])

array([ True])

**✍️ Ringkasan Temuan Eksplorasi:**

*(Kolom apa yang bermasalah di setiap tabel, seberapa parah, dan prioritas penanganan kamu)*

Hasil yang kami peroleh dari **Data Cleaning dan Exploration**:
- Pada kolom ***`internal_credit_score`*** di data ***users*** terdapat missing value sebanyak **15%** atau sekitar **5250 users**.
- Terdapat inkonsistensi penamaan ***`payment_method`*** pada data ***transaction*** yang perlu **distandarisasi**.
- Nominal anomali yang bernilai **-15000 (75 baris)** dan **999999999 (76 baris)** pada kolom ***`late_fee`*** di data ***transaction***
> 

---
### 2.2 Kerangka Identifikasi Missing Values

Sebelum menangani missing values pada kolom manapun, identifikasi dulu **jenis missing value-nya**.

| Jenis | Definisi Singkat | Implikasi Penanganan | Contoh di Dataset Ini |
|---|---|---|---|
| **MCAR** *(Missing Completely At Random)* | Nilai kosong tidak berkaitan dengan kolom lain. Pola missing benar-benar acak. | Relatif aman di-impute atau di-drop tanpa bias signifikan. | Sebagian kecil `internal_credit_score` kosong karena gangguan sistem scraping acak. |
| **MAR** *(Missing At Random)* | Nilai kosong berkaitan dengan kolom **lain**, bukan dengan nilai kolom itu sendiri. | Imputation berbasis kolom lain lebih tepat. Drop bisa menyebabkan bias. | `internal_credit_score` kosong mungkin berkorelasi dengan `gopay_tier` atau `join_date`. |
| **MNAR** *(Missing Not At Random)* | Nilai kosong berkaitan langsung dengan nilai yang seharusnya ada. Ada alasan sistematis. | Imputation apapun berisiko misleading. Perlu keputusan bisnis eksplisit. | `internal_credit_score` kosong justru karena user tidak pernah bertransaksi — nilai kosong itu sendiri adalah sinyal risiko. |

> 💡 **Cara Menggunakan Kerangka Ini:**
> Untuk setiap kolom bermasalah, tanyakan:
> 1. Apakah pola missing-nya acak, atau ada pola tertentu?
> 2. Apakah nilai kosong berkaitan dengan kolom lain?
> 3. Apakah nilai kosong itu sendiri mengandung informasi bisnis?
>
> Justifikasi reasoning kamu lebih penting dari labelnya.

In [30]:
dfMerge_tu.loc[:, dfMerge_tu.notnull().any(axis = 0)]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,GoPay
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,GoPay
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,PayLater
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,GoPay


---
### 2.3 Penanganan Kolom `payment_method`

Kolom ini memiliki 8 varian penulisan untuk 3 metode pembayaran yang berbeda, akibat inkonsistensi penamaan antar micro-service.

| Varian Asli | Metode Sebenarnya |
|---|---|
| `GoPay`, `gopay`, `GO-PAY` | GoPay (saldo digital) |
| `GoPayLater`, `PayLater`, `gopay_later` | GoPayLater (kredit) |
| `Cash`, `CASH` | Cash |

> 🧠 **Critical Thinking Prompt:**  
> Setelah standarisasi, periksa ulang: apakah ada user **Basic tier** yang menggunakan GoPayLater?  
> Secara aturan bisnis, PayLater hanya boleh digunakan oleh user Plus.  
> Jika ada, apakah itu error data atau bug sistem validasi?

In [31]:
# Lihat semua nilai unik di payment_method beserta frekuensinya
pm_unique = df_trx["payment_method"].unique()
pm_valcount = df_trx["payment_method"].value_counts()
display(pm_unique, pm_valcount)

<ArrowStringArray>
[      'GoPay',       'gopay',    'PayLater',  'GoPayLater',      'GO-PAY',
        'Cash', 'gopay_later',        'CASH']
Length: 8, dtype: str

payment_method
GoPay          90240
GoPayLater     60014
gopay          59854
PayLater       30017
GO-PAY         15136
Cash           14978
CASH           14972
gopay_later    14789
Name: count, dtype: int64

**✍️ Mapping standarisasi yang kamu buat:**
- Varian asli → nilai standar (tuliskan mapping lengkapnya):
- Format standar yang kamu pilih dan alasannya:
- Temuan setelah standarisasi (apakah ada Basic tier yang pakai GoPayLater?):

> 

In [32]:
# TODO: Standarisasi payment_method
# Simpan hasil ke kolom baru: payment_method_clean

dfMerge_tu["payment_method_clean"] = dfMerge_tu["payment_method"].replace({
    "gopay": "GoPay",
    "GO-PAY": "GoPay",
    "GoPayLater": "PayLater",
    "gopay_later": "PayLater",
    "CASH": "Cash"
})

display(dfMerge_tu)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,GoPay
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,GoPay
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,PayLater
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,GoPay


In [33]:
# Verifikasi: cek Basic tier yang menggunakan GoPayLater setelah standarisasi
tier_paylater = dfMerge_tu[dfMerge_tu["payment_method"] == "GoPayLater"]["gopay_tier"].value_counts()
display(tier_paylater)

gopay_tier
Plus     36138
Basic    23876
Name: count, dtype: int64

---
### 2.4 Penanganan `internal_credit_score`

Kolom `internal_credit_score` di tabel users memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

> 🧠 **Critical Thinking Prompt:**  
> Apakah nilai kosong ini karena sistem gagal mencatat, atau karena user memang belum punya histori kredit?  
> User tanpa credit score = *unscored* — di industri fintech, ini dianggap risiko tersendiri.  
> Keputusan kamu di sini akan langsung mempengaruhi hasil analisis profil risiko di Section 4.

In [34]:
# Investigasi pola missing values pada internal_credit_score
# Apakah berkorelasi dengan gopay_tier, paylater_limit, atau join_date?

# Display tabel dimana rows adalah semua df_users yang memiliki credit score kosong, dan berkolom gopay_tier, paylater_limit, atau join_date
# Karena tier basic sudah dipastikan akan punya paylater limit = 0, maka untuk instance ini diexclude saja.
result = df_users.loc[
    df_users["internal_credit_score"].isnull() & (df_users["gopay_tier"] != "Basic"),
    ["gopay_tier", "paylater_limit", "join_date"]
]
result.head(15)


# Basic -> paylaterlimit = 0
# plus -> paylaterlimit = 500.000, 5.000.000, 3.000.000, 500.000, 1.500.000

,gopay_tier,paylater_limit,join_date
7,Plus,500000,2023-03-26
9,Plus,5000000,2021-04-20
19,Plus,3000000,2022-06-23
25,Plus,500000,2022-01-15
35,Plus,1500000,2022-07-20
39,Plus,5000000,2023-04-27
77,Plus,1500000,2021-03-22
102,Plus,3000000,2022-07-13
116,Plus,1500000,2023-04-19
119,Plus,500000,2022-01-27


In [35]:
# Cek: apakah user yang credit_score-nya missing lebih banyak yang default?
# Hint: merge dengan df_trx, lalu bandingkan default rate

#Buat dataframe baru yang merupakan merge dari users dan transaction. Mirip dengan dfMerge namun hanya menggabungkan internal_credit_score dari df_users
dfUserTrx = df_users[["user_id", "internal_credit_score"]].merge(df_trx,on="user_id", how="left")

In [38]:
# Unique user counts
null_score_users = dfUserTrx[dfUserTrx['internal_credit_score'].isna()]['user_id'].nunique()
default_null_score_users = dfUserTrx[
    (dfUserTrx['internal_credit_score'].isna()) & 
    (dfUserTrx['payment_status'] == 'Default')
]['user_id'].nunique()

print(f"Users dengan NaN Credit Score: {null_score_users:,}")
print(f"Users dengan NaN Credit Score & Default: {default_null_score_users:,}")
print(f"Default Rate antar NaN Users: {default_null_score_users / null_score_users:.2%}")

Users dengan NaN Credit Score: 5,250
Users dengan NaN Credit Score & Default: 1,860
Default Rate antar NaN Users: 35.43%


**✍️ Analisis & Justifikasi:**
- **Jenis missing value (MCAR / MAR / MNAR):** dan alasan klasifikasi kamu:
- Temuan investigasi pola missing (berkorelasi dengan tier? join_date?):
- Apakah user tanpa credit score memiliki default rate yang berbeda?
- Keputusan penanganan (drop / impute / pertahankan NaN) dan alasan:

> 

***Penganan internal_credit_score = NaN***

Kolom ***`internal_credit_score`*** di tabel ***users*** memiliki **5.250 nilai kosong (~15%)** — variabel kritis untuk analisis risiko kredit.

**Jenis missing value**:\
MCAR.

**Pola missing**:\
Tidak ditemukan korelasi apapun dengan kolom *join_date*, *gopay_tier*, maupun *paylaterlimit*.

**Apakah user tanpa credit score memiliki default rate yang beda**:\
Ya, berbeda. Namun Default rate dalam user tanpa credit score adalah **35.43%**, sehingga tidak terlalu bisa dikorelasikan dengan ketidakadaan nilai *internal_credit_score*.

**Keputusan akhir**:\
Di-impute menjadi 0. Kami tidak berencana untuk dropping karena dengan itu kita akan kehilangan **15%** data.

In [41]:
# TODO: Implementasi penanganan missing values internal_credit_score
(df_users["internal_credit_score"] == 0).sum()

np.int64(0)

In [42]:
#Making sure bahwa semua yang ber-tier Basic pasti tidak punya paylater dan internal_credit_score
dfBasicNan = dfMerge_tu.loc[dfMerge_tu["internal_credit_score"].isnull() & (dfMerge_tu["gopay_tier"] == "Basic")]
display(dfBasicNan)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater
...,...,...,...,...,...,...,...,...,...,...,...,...,...
299923,GP-034992,2022-10-19,Basic,NaN,0,GTRX-0153499,SVC-004,2023-01-02 03:00:00,gopay,529392.00,0.00,Paid,GoPay
299924,GP-034992,2022-10-19,Basic,NaN,0,GTRX-0191552,SVC-016,2023-03-10 04:00:00,GoPay,70341.00,0.00,Paid,GoPay
299925,GP-034992,2022-10-19,Basic,NaN,0,GTRX-0207754,SVC-019,2023-04-20 13:00:00,GoPay,286566.00,0.00,Paid,GoPay
299926,GP-034992,2022-10-19,Basic,NaN,0,GTRX-0214808,SVC-019,2023-06-26 19:00:00,GoPayLater,389409.00,0.00,Paid,PayLater


In [44]:
# All with Basic tiers will have
(dfBasicNan["gopay_tier"] == "Basic").sum()

np.int64(18222)

In [46]:
dfMerge_tu.head(5)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater


In [47]:
df_users.loc[(df_users["gopay_tier"] == "Basic") & (df_users["paylater_limit"] != 0)]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit


---
### 2.5 Penanganan `late_fee`

Kolom `late_fee` memiliki dua jenis anomali yang berbeda sifatnya — tangani secara terpisah.

> 🧠 **Critical Thinking Prompt:**  
> Di industri fintech, denda keterlambatan diatur oleh regulasi OJK.  
> Nilai `late_fee` yang sangat besar bisa berarti bug sistem, bukan kebijakan yang valid.  
> Keputusan kamu harus mempertimbangkan aspek **compliance**, bukan hanya statistik.

In [227]:
# Investigasi distribusi late_fee secara menyeluruh
# Berapa nilai negatif? Berapa nilai ekstrem?

# Nilai negatif
negative_late = df_trx["late_fee"] < 0

print("Jumlah nilai negatif:", negative_late.sum())
print("Nilai negatif:", df_trx.loc[negative_late, "late_fee"].unique())
print()

# Nilai ekstrem (Interquartile Range)
extreme_late = df_trx["late_fee"] > 10_000_000

print("Jumlah nilai ekstrem:", extreme_late.sum())
print("Range nilai ekstrem:", df_trx.loc[extreme_late, "late_fee"].min(), "hingga", df_trx.loc[extreme_late, "late_fee"].max())

Jumlah nilai negatif: 75
Nilai negatif: [-15000.]

Jumlah nilai ekstrem: 76
Range nilai ekstrem: 99999999.0 hingga 99999999.0


In [229]:
# Anomali 1: Nilai negatif
# Apakah terjadi pada payment_status tertentu?
negative_val = df_trx[negative_late]

display(negative_val[["late_fee", "payment_status"]])
display(negative_val["payment_status"].value_counts())

,late_fee,payment_status
1586,-15000.00,Default
7772,-15000.00,Default
10931,-15000.00,Default
15832,-15000.00,Default
15853,-15000.00,Default
...,...,...
260771,-15000.00,Default
273060,-15000.00,Default
276295,-15000.00,Default
277153,-15000.00,Default


payment_status
Default    75
Name: count, dtype: int64

In [237]:

# Anomali 2: Nilai ekstrem tinggi (> Rp10 juta)
# Apakah ada pola pada service atau user tertentu?

extreme_val = df_trx[extreme_late]

display(extreme_val[["late_fee", "user_id", "service_id"]])
display(extreme_val["service_id"].value_counts())
display(extreme_val["user_id"].value_counts())
print("Unique users:", extreme_val["user_id"].nunique())

,late_fee,user_id,service_id
7841,99999999.00,GP-022226,SVC-008
8739,99999999.00,GP-003899,SVC-006
12362,99999999.00,GP-025589,SVC-020
15722,99999999.00,GP-024468,SVC-019
16360,99999999.00,GP-022374,SVC-002
...,...,...,...
285740,99999999.00,GP-006536,SVC-009
293899,99999999.00,GP-030129,SVC-014
294317,99999999.00,GP-012269,SVC-020
294593,99999999.00,GP-004582,SVC-019


service_id
SVC-015    7
SVC-019    6
SVC-012    6
SVC-020    5
SVC-011    5
SVC-007    5
SVC-004    5
SVC-010    5
SVC-008    4
SVC-014    4
SVC-006    3
SVC-002    3
SVC-001    3
SVC-005    3
SVC-013    3
SVC-009    3
SVC-003    3
SVC-016    3
Name: count, dtype: int64

user_id
GP-022226    1
GP-003899    1
GP-025589    1
GP-024468    1
GP-022374    1
            ..
GP-006536    1
GP-030129    1
GP-012269    1
GP-004582    1
GP-012663    1
Name: count, Length: 76, dtype: int64

Unique users: 76


In [233]:
extreme_analysis = (extreme_val.groupby("service_id").agg(anomaly_count=("late_fee", "count"), unique_users=("user_id", "nunique")).reset_index())
extreme_analysis = extreme_analysis.merge(df_services[["service_id", "service_name", "category"]], on="service_id", how="left")
extreme_analysis = extreme_analysis.sort_values("unique_users", ascending=False)

display(extreme_analysis)

,service_id,anomaly_count,unique_users,service_name,category
14,SVC-015,7,7,Merchant Offline A,Offline QRIS
11,SVC-012,6,6,Game Voucher,Digital Goods & Bills
16,SVC-019,6,6,Merchant Offline E,Offline QRIS
6,SVC-007,5,5,GoBox,Logistics
10,SVC-011,5,5,BPJS,Digital Goods & Bills
3,SVC-004,5,5,GoFood,Food Delivery
9,SVC-010,5,5,PDAM,Digital Goods & Bills
17,SVC-020,5,5,Merchant Offline F,Offline QRIS
7,SVC-008,4,4,Pulsa/Data,Digital Goods & Bills
13,SVC-014,4,4,Investasi,Digital Goods & Bills


Terdapat 76 transaksi di 18 layanan yang memiliki nilai late_fee melebihi Rp10 juta. Anomali-anomali tersebut tersebar di seluruh lima kategori layanan, dengan konsentrasi tertinggi pada kategori Barang Digital & Tagihan (30 kasus) dan QRIS Offline (21 kasus). Merchant Offline A memiliki jumlah transaksi anomali terbanyak di tingkat layanan, yaitu 7 transaksi. Setiap layanan yang terpengaruh menunjukkan jumlah transaksi anomali dan pengguna unik yang sama, yang mengindikasikan tidak adanya anomali berulang oleh pengguna yang sama dalam satu layanan. Validasi lebih lanjut diperlukan untuk menentukan apakah nilai-nilai ekstrem tersebut sah atau disebabkan oleh kesalahan data/sistem. 

**✍️ Analisis & Justifikasi — Anomali 1 (late_fee negatif):**
- **Jumlah baris terdampak:**\
**75 baris**, seluruh 75 transaksi memiliki **late_fee = -15000**
- **Hipotesis penyebab:**\
*late_fee* adalah denda keterlambatan, nilai negatif pada *late_fee* secara logika tidak wajar. Nilai negatif pada *late_fee* seragam dan seluruh **`payment_status`** bernilai ***Default*** yang dimana dapat mengindikasikan kepada ***logical error***.
- **Keputusan penanganan dan alasan:**\
Data perlu di **flag** terlebih dahulu untuk di validasi pada aturan bisnis dan sumber transaksi. Jika data diubah ke nilai *absolut(abs)*, nilai -15000 akan berubah menjadi 15000 yang dimana akan mengubah makna data. 
>

**✍️ Analisis & Justifikasi — Anomali 2 (late_fee ekstrem):**
- **Jumlah baris terdampak dan range nilainya:**\
**76 baris** dengan late_fee > 10.000.000, dengan range **nilai minimum = 999.999.999**, dan **nilai maksimum = 999.999.999**
- **Threshold yang kamu pilih untuk mendefinisikan 'ekstrem' dan alasannya:**\
Angka 10.000.000 dipilih untuk mengidentifikasi ***`late_fee`*** yang sangat tinggi. Hasil menunjukan **76 transaksi** tersebut memiliki **nilai identik 999.999.999**.
- **Hipotesis penyebab:**\
Nilai minimum dan maksimum bernilai 999.999.999 yang mengindikasikan bug pada sistem perhitungan
- **Keputusan penanganan dan alasan:**\
Nilai 999.999.999 muncul pada **18 layanan** dan **76 transaksi user yang berbeda**. Anomali-anomali tersebut tersebar di seluruh kategori layanan, dengan konsentrasi tertinggi pada kategori ***Barang Digital & Tagihan*** (30 kasus) dan ***QRIS Offline*** (21 kasus). kemungkinan besar dikarenakan system issue, bukan karena user. Kami putuskan untuk **flag**.

> 

In [ ]:
# TODO: Implementasi penanganan Anomali 1 (late_fee negatif)
df_trx["late_fee_negative_flag"] = negative_late

In [ ]:
# TODO: Implementasi penanganan Anomali 2 (late_fee ekstrem)
df_trx["late_fee_extreme_flag"] = extreme_late

In [318]:
df_trx["late_fee_anomaly"] = (
    df_trx["late_fee_negative_flag"] |
    df_trx["late_fee_extreme_flag"])
display(df_trx)

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,late_fee_negative_flag,late_fee_extreme_flag,late_fee_anomaly,is_paylater_violation,service_category,is_default
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False,False,False,False,Digital Goods & Bills,False
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False,False,False,False,Digital Goods & Bills,False
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False,False,False,False,Mobility,False
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False,False,False,False,Mobility,False
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False,False,False,False,Mobility,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False,False,False,False,Mobility,False
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False,False,False,False,Offline QRIS,False
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False,False,False,False,Food Delivery,False
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False,False,False,False,Offline QRIS,False


In [319]:
print("Total anomali yang di flag:", df_trx["late_fee_anomaly"].sum())

Total anomali yang di flag: 151


In [213]:
df_trx["flagged_anomali"]

df_trx[df_trx["late_fee"] != 0]["late_fee"].describe()

count      15577.00
mean      514893.58
std      6966260.82
min       -15000.00
25%        15955.00
50%        27239.00
75%        38669.00
max     99999999.00
Name: late_fee, dtype: float64

---
### 2.6 Penanganan Anomali Tanggal: `trx_date` sebelum `join_date`

Terdapat **~30.177 transaksi (~10%)** dengan `trx_date` lebih awal dari `join_date` user — secara logika bisnis tidak mungkin terjadi.

> 🧠 **Critical Thinking Prompt:**  
> Di konteks fintech, transaksi sebelum akun dibuat bisa mengindikasikan **fraud** atau **data migration issue**.  
> Drop vs. flag memiliki implikasi berbeda: drop menghilangkan sinyal fraud, flag mempertahankannya untuk analisis.  
> Apakah anomali ini lebih banyak terjadi pada user yang akhirnya **Default**?

In [48]:
# Konversi kolom tanggal ke datetime
df_trx['trx_date'] = pd.to_datetime(df_trx['trx_date'])
df_users['join_date'] = pd.to_datetime(df_users['join_date'])

In [50]:
# Identifikasi transaksi dengan trx_date < join_date
# Investigasi: seberapa besar selisih tanggalnya? Distribusi selisih negatif?
df_merged = df_trx.merge(df_users, on='user_id', how='left')

# hitung selisih
df_merged['date_diff_days'] = (df_merged['trx_date'] - df_merged['join_date']).dt.days
df_merged['date_diff_days'].value_counts() # selisih tanggal mulai dari 160days sampai 720days

# cari data anomali
df_anomali = df_merged[df_merged['trx_date'] < df_merged['join_date']] # get yang less than aja
df_merged['is_anomali_tanggal'] = df_merged['date_diff_days'] < 0 # get yang negatif
#print(df_anomali[['trx_id', 'user_id', 'trx_date', 'join_date', 'date_diff_days']])

display(df_anomali['date_diff_days'].describe())

count   30177.00
mean      -89.90
std        63.30
min      -269.00
25%      -135.00
50%       -79.00
75%       -37.00
max        -1.00
Name: date_diff_days, dtype: float64

In [53]:
# Apakah anomali ini berkorelasi dengan payment_status = Default?
# Apakah tersebar merata atau terkonsentrasi pada user/tanggal tertentu?
from scipy.stats import chi2_contingency
# cek korelasi:
crosstab_status1 = pd.crosstab(df_merged['is_anomali_tanggal'], df_merged['payment_status'], normalize='index')
crosstab_status2 = pd.crosstab(df_merged['is_anomali_tanggal'], df_merged['payment_method_clean'], normalize='index')
crosstab_status3 = pd.crosstab(df_merged['is_anomali_tanggal'], df_merged['user_id'], normalize='index')
crosstab_raw = pd.crosstab(df_merged['is_anomali_tanggal'], df_merged['trx_date'])

# 2. Jalankan Uji Chi-Square
chi2, p_value, dof, expected = chi2_contingency(crosstab_status1)

print(f"P-Value: {p_value}")
display(crosstab_status1)

#cek anomali per bulan
anomali_per_bulan = df_anomali['trx_date'].dt.to_period('M').value_counts().sort_index()
display(anomali_per_bulan)
# df_merged
# Ditemukan: tidak terdapat korelasi dengan payment status, user id maupun trx_date tertentu
# Tapi ada sedikit pola per bulan, anomali konsisten menurun, Januari - September, 6000 - 200an

P-Value: 0.9999910782883896


payment_status,Default,Paid,Pending
is_anomali_tanggal,,,
False,0.05,0.91,0.04
True,0.05,0.91,0.03


trx_date
2023-01    6709
2023-02    5211
2023-03    4937
2023-04    4193
2023-05    3381
2023-06    2512
2023-07    1859
2023-08    1106
2023-09     269
Freq: M, Name: count, dtype: int64

**✍️ Analisis & Justifikasi:**
- **Jumlah baris terdampak dan distribusi selisih tanggal:**\
**30,177 baris** terdampak anomali
- **Jenis anomali (acak / berpola):**\
Anomali berpola per bulan dari januari 2023 di 6000 transaksi menurun di september 2023 pada 200an transaksi.
- **Hipotesis penyebab (migration error? clock skew? fraud?):**\
Migration error. karena menurun secara sistematis. dan tidak ada korelasi pada user/payment status.
- **Apakah anomali ini berkorelasi dengan Default? Implikasi untuk analisis risiko:**\
Tidak berkorelasi, p 0.6 tidak signifikan
- **Keputusan penanganan (drop / flag / pertahankan) dan alasan:**\
**Flag**, untuk konfirmasi ke tim engineering

> 

In [59]:
# TODO: Implementasi penanganan anomali tanggal
dfMerge_tu['is_anomali_tanggal'] = df_merged['date_diff_days'] < 0
display(df_trx)

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False
...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False


---
### 2.7 Penanganan Business Logic Error: Transaksi PayLater Melebihi Limit

**Ini adalah anomali paling kritis di dataset ini.** Lebih dari 50% transaksi GoPayLater memiliki `amount` yang melebihi `paylater_limit` user, termasuk user Basic tier yang seharusnya tidak punya PayLater sama sekali.

| Tipe Pelanggaran | Deskripsi |
|---|---|
| **Basic tier pakai PayLater** | User dengan `paylater_limit = 0` bertransaksi dengan GoPayLater |
| **Plus tier melebihi limit** | User PayLater sah, tapi `amount > paylater_limit` |
| **Transaksi valid** | User Plus dengan `amount ≤ paylater_limit` |

> 🧠 **Critical Thinking Prompt:**  
> Jangan drop transaksi over-limit — ini adalah **data paling berharga** untuk memahami bug dan pola default.  
> Pertahankan dengan flag, lalu analisis secara terpisah.  
> **Dropping = menghilangkan bukti.**

In [62]:
# Merge transaksi GoPayLater dengan data paylater_limit user
# Identifikasi tipe pelanggaran untuk setiap transaksi
df27 = df_users.merge(df_trx, on="user_id", how="left")
display(df27)
df27["paylater_limit"].value_counts()

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay,False
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay,False
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay,False
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay,False
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,GoPay,False
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,GoPay,False
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,PayLater,False
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,GoPay,False


paylater_limit
0          119144
500000      72234
1500000     54281
3000000     36791
5000000     17558
Name: count, dtype: int64

In [65]:
# Kuantifikasi: berapa jumlah dan total nilai (Rupiah) dari setiap tipe pelanggaran?
total_exceedPayLater = df27.loc[df27["paylater_limit"] > df27["amount"], "amount"].sum()
display(total_exceedPayLater)

total_zeroPayLater = df27.loc[df27["paylater_limit"] == 0, "amount"].sum()
display(total_zeroPayLater)

total_ZeroExceed = df27.loc[(df27["paylater_limit"] == 0) & (df27["amount"] > df27["paylater_limit"]), "amount"].sum()
display(total_ZeroExceed)

np.float64(54937772859.0)

np.float64(49663530665.0)

np.float64(49663530665.0)

In [71]:
def flag(row):
    if (row["paylater_limit"] == 0) & (row["amount"] > 0):
        return "unauthorized_paylater"
    elif row["amount"] > row["paylater_limit"]:
        return "over_limit_paylater"
    elif (row["paylater_limit"] > row["amount"]):
        return "normal"

df27["flag"] = df27.apply(flag, axis=1)
display(df27)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal,flag
0,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0003439,SVC-001,2023-08-31 17:00:00,GoPay,704068.00,0.00,Paid,GoPay,False,unauthorized_paylater
1,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0137991,SVC-020,2023-08-15 06:00:00,GO-PAY,195808.00,0.00,Paid,GoPay,False,unauthorized_paylater
2,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0172755,SVC-013,2023-10-08 01:00:00,gopay,606859.00,0.00,Paid,GoPay,False,unauthorized_paylater
3,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0259404,SVC-019,2023-03-22 00:00:00,GoPay,356899.00,0.00,Paid,GoPay,False,unauthorized_paylater
4,GP-000001,2022-05-26,Basic,NaN,0,GTRX-0282350,SVC-007,2023-02-08 21:00:00,GoPayLater,186928.00,0.00,Paid,PayLater,False,unauthorized_paylater
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300003,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0188082,SVC-019,2023-09-04 03:00:00,GoPay,236650.00,0.00,Paid,GoPay,False,unauthorized_paylater
300004,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0194534,SVC-017,2023-07-06 19:00:00,gopay,138743.00,0.00,Paid,GoPay,False,unauthorized_paylater
300005,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0199113,SVC-004,2023-08-06 15:00:00,GoPayLater,749400.00,0.00,Paid,PayLater,False,unauthorized_paylater
300006,GP-035000,2021-07-27,Basic,362.00,0,GTRX-0232900,SVC-008,2023-02-28 21:00:00,gopay,568217.00,0.00,Paid,GoPay,False,unauthorized_paylater


In [73]:
df27.loc[df27["flag"] == "normal", :]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal,flag
6,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0017474,SVC-010,2023-06-29 12:00:00,gopay,200782.00,0.00,Paid,GoPay,True,normal
7,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0085613,SVC-007,2023-08-14 06:00:00,GoPay,251279.00,0.00,Paid,GoPay,False,normal
8,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0089307,SVC-008,2023-01-24 20:00:00,GoPay,191460.00,0.00,Paid,GoPay,True,normal
9,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0101411,SVC-003,2023-05-24 07:00:00,gopay,88206.00,0.00,Paid,GoPay,True,normal
10,GP-000002,2023-07-12,Plus,751.00,500000,GTRX-0103354,SVC-007,2023-05-18 08:00:00,GoPayLater,357436.00,37815.00,Default,PayLater,True,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
299982,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0165566,SVC-016,2023-03-17 16:00:00,GoPay,539583.00,0.00,Paid,GoPay,True,normal
299983,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0187097,SVC-018,2023-05-06 13:00:00,gopay,708976.00,0.00,Paid,GoPay,True,normal
299984,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0209452,SVC-014,2023-02-28 07:00:00,GoPay,686240.00,0.00,Paid,GoPay,True,normal
299985,GP-034998,2023-08-06,Plus,665.00,1500000,GTRX-0211865,SVC-003,2023-11-24 13:00:00,GoPayLater,172077.00,13120.00,Default,PayLater,False,normal


In [74]:
df27.loc[df27["flag"].isnull(), :]

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,trx_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal,flag
12962,GP-001524,2021-01-27,Plus,472.00,500000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75508,GP-008804,2021-05-11,Basic,492.00,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
160580,GP-018739,2021-10-08,Plus,745.00,500000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201812,GP-023538,2021-04-07,Plus,NaN,500000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
209201,GP-024403,2021-03-07,Basic,323.00,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
228993,GP-026708,2021-06-13,Basic,406.00,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
251878,GP-029370,2022-06-23,Basic,NaN,0,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
279902,GP-032647,2022-05-22,Plus,438.00,500000,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [75]:
(df27["amount"] == 0).sum()

np.int64(0)

In [ ]:
# Kritis: apakah transaksi over-limit berkorelasi dengan payment_status = Default?
# Bandingkan default rate antara: transaksi valid vs over-limit


**✍️ Analisis & Justifikasi:**
- Jumlah dan persentase setiap tipe pelanggaran:
- Total nilai Rupiah yang terlibat dalam pelanggaran:
- Apakah over-limit berkorelasi dengan Default? Temuan kamu:
- Keputusan penanganan (flag, bukan drop) dan kolom flag yang kamu buat:
- Hipotesis mengapa bug ini bisa terjadi di sistem:

> 

In [ ]:
# TODO: Buat kolom flag untuk tipe pelanggaran PayLater
# Contoh: 'valid', 'over_limit', 'unauthorized'


---
### 2.8 Penanganan Duplikat & Integritas Data

In [290]:
# 1. Cek exact duplicates di setiap tabel
def dupl(df):
    result = {}
    for i in df.columns:
        result[i] = df[i].duplicated().sum()
    return pd.Series(result)

display(dupl(df_users))
display(dupl(df_services))
display(dupl(df_trx))

user_id                      0
join_date                34000
gopay_tier               34998
internal_credit_score    34449
paylater_limit           34995
dtype: int64

service_id       0
service_name     0
category        15
dtype: int64

trx_id                         0
user_id                   265008
service_id                299980
trx_date                  291240
payment_method            299992
amount                     49384
late_fee                  286931
payment_status            299997
payment_method_clean      299997
late_fee_negative_flag    299998
late_fee_extreme_flag     299998
late_fee_anomaly          299998
dtype: int64

In [287]:
# 2. Cek duplikat trx_id
df_dupl = df_trx["trx_id"].duplicated().sum()
display(df_dupl)

np.int64(0)

In [279]:
# 3. Cek service_id di transactions yang tidak ada di services
dfMerge = pd.merge(df_trx, df_services[["service_id"]], on="service_id", how="left", indicator="duplicate")
df_services_cek = dfMerge[dfMerge["duplicate"] == "left_only"]
display(df_services_cek)
display((dfMerge["duplicate"] == "left_only").sum())

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,late_fee_negative_flag,late_fee_extreme_flag,late_fee_anomaly,duplicate


np.int64(0)

In [282]:
# 4. Cek user_id di transactions yang tidak ada di users
dfMerge = pd.merge(df_trx, df_users[["user_id"]], on="user_id", how="left", indicator="duplicate")
df_services_cek = dfMerge[dfMerge["duplicate"] == "left_only"]
display(df_services_cek)
display((dfMerge["duplicate"] == "left_only").sum())

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,late_fee_negative_flag,late_fee_extreme_flag,late_fee_anomaly,duplicate


np.int64(0)

In [281]:
# 5. Cek inkonsistensi logika: payment_status = Pending tapi late_fee > 0
df_trx_inkonsisten = ((df_trx["payment_status"] == "Pending") & (df_trx["late_fee"] > 0))
display(df_trx[df_trx_inkonsisten])
display(df_trx_inkonsisten.sum())

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,late_fee_negative_flag,late_fee_extreme_flag,late_fee_anomaly


np.int64(0)

**✍️ Analisis & Justifikasi:**
- Masalah yang ditemukan dan jumlah baris terdampak: 0, tidak ada duplikasi pada dataset
- Hipotesis untuk setiap masalah: Dataset yang digunakan tidak memiliki duplikasi alias nilainya unik
- Keputusan penanganan per masalah: -
> 

In [ ]:
# TODO: Implementasi keputusan penanganan masalah integritas


---
## 3. Feature Engineering

### 3.1 Fitur Wajib

Buat 8 kolom berikut. Sertakan penjelasan singkat business value-nya di setiap fitur.

#### ⚙️ `payment_method_clean`
*(Sudah dibuat di Section 2.3 — pastikan sudah ada di df_trx)*

In [76]:
df_trx["payment_method_clean"]

0            GoPay
1            GoPay
2            GoPay
3            GoPay
4         PayLater
            ...   
299995        Cash
299996       GoPay
299997    PayLater
299998       GoPay
299999       GoPay
Name: payment_method_clean, Length: 300000, dtype: str

#### ⚙️ `credit_score_tier`

> 💡 Default threshold: Poor (300–499), Fair (500–649), Good (650–749), Excellent (750–850).  
> Sesuaikan jika analisis distribusi kamu menunjukkan pembagian yang lebih bermakna secara bisnis.

**✍️ Threshold yang kamu pilih dan alasannya:**

> 

In [78]:
# TODO: Buat credit_score_tier di df_users
# Pertimbangkan: bagaimana menangani user yang credit_score-nya NaN?
def tier(score):
    if pd.isna(score):
        return "None"
    elif score < 500:
        return "Poor"
    elif score < 650:
        return "Fair"
    elif score < 750:
        return "Good"
    elif score < 850:
        return "Very Good"
    else:
        return "Excellent"

df_users["credit_score_tier"] = df_users["internal_credit_score"].apply(tier)

display(df_users)

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,is_paylater_violation,credit_score_tier
0,GP-000001,2022-05-26,Basic,NaN,0,False,None
1,GP-000002,2023-07-12,Plus,751.00,500000,False,Very Good
2,GP-000003,2022-05-09,Plus,728.00,1500000,False,Good
3,GP-000004,2021-08-28,Basic,394.00,0,False,Poor
4,GP-000005,2021-05-31,Basic,333.00,0,True,Poor
...,...,...,...,...,...,...,...
34995,GP-034996,2022-03-12,Plus,544.00,1500000,False,Fair
34996,GP-034997,2023-07-23,Plus,NaN,5000000,False,None
34997,GP-034998,2023-08-06,Plus,665.00,1500000,False,Good
34998,GP-034999,2022-05-29,Basic,391.00,0,False,Poor


#### ⚙️ `is_paylater_violation`

> 💡 Perlu merge df_trx dengan df_users untuk mendapatkan paylater_limit per transaksi.

In [77]:
# TODO: Buat is_paylater_violation (boolean)
# True jika payment_method_clean == 'gopaylater' AND amount > paylater_limit
dfMerge_tu = pd.merge(df_users, df_trx, on="user_id", how="left")

df_users["is_paylater_violation"] = (
    (dfMerge_tu["payment_method_clean"] == "PayLater") &
    (dfMerge_tu["amount"] > dfMerge_tu["paylater_limit"])
)

display(df_users)
display(df_users["is_paylater_violation"].sum())

,user_id,join_date,gopay_tier,internal_credit_score,paylater_limit,is_paylater_violation
0,GP-000001,2022-05-26,Basic,NaN,0,False
1,GP-000002,2023-07-12,Plus,751.00,500000,False
2,GP-000003,2022-05-09,Plus,728.00,1500000,False
3,GP-000004,2021-08-28,Basic,394.00,0,False
4,GP-000005,2021-05-31,Basic,333.00,0,True
...,...,...,...,...,...,...
34995,GP-034996,2022-03-12,Plus,544.00,1500000,False
34996,GP-034997,2023-07-23,Plus,NaN,5000000,False
34997,GP-034998,2023-08-06,Plus,665.00,1500000,False
34998,GP-034999,2022-05-29,Basic,391.00,0,False


np.int64(6207)

#### ⚙️ `paylater_usage_ratio`

> 💡 Hanya relevan untuk transaksi GoPayLater. Untuk transaksi non-PayLater, isi dengan NaN.

In [295]:
# TODO: Buat paylater_usage_ratio (amount / paylater_limit)
# Handle division by zero untuk user dengan paylater_limit = 0


,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,late_fee_negative_flag,late_fee_extreme_flag,late_fee_anomaly
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False,False,False
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False,False,False
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False,False,False
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False,False,False
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False,False,False
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False,False,False
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False,False,False
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False,False,False


#### ⚙️ `user_tenure_days`

> 💡 Tentukan sendiri tanggal referensi yang kamu gunakan dan justifikasikan.

**✍️ Tanggal referensi yang kamu gunakan dan alasannya:**

> 

In [ ]:
# TODO: Buat user_tenure_days di df_users


#### ⚙️ `has_late_fee`

In [ ]:
# TODO: Buat has_late_fee (boolean: True jika late_fee > 0)
# Pastikan menggunakan late_fee yang sudah di-clean dari Section 2.5
# late_fee_anomaly
df_trx["has_late_fee"] = df_trx["late_fee_anomaly"]

#### ⚙️ `is_default`

In [79]:
# TODO: Buat is_default (boolean: True jika payment_status == 'Default')
df_trx["is_default"] = df_trx["payment_status"] == "Default"
display(df_trx)

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal,is_default
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False,False
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False,False
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False,False
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False,False
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False,False
...,...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False,False
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False,False
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False,False
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False,False


#### ⚙️ `service_category`

> 💡 Join df_trx dengan df_services untuk mendapatkan kategori layanan per transaksi.

In [80]:
# TODO: Buat service_category dengan merge ke df_services
dfMerge_ts = pd.merge(df_trx, df_services, on="service_id", how="left")
df_trx["service_category"] = dfMerge_ts["category"]
display(df_trx)
display(df_trx["service_category"].value_counts())

,trx_id,user_id,service_id,trx_date,payment_method,amount,late_fee,payment_status,payment_method_clean,is_anomali_tanggal,is_default,service_category
0,GTRX-0000001,GP-018595,SVC-014,2023-03-10 06:00:00,GoPay,157588,0.00,Paid,GoPay,False,False,Digital Goods & Bills
1,GTRX-0000002,GP-000244,SVC-014,2023-12-12 04:00:00,GoPay,594659,0.00,Paid,GoPay,False,False,Digital Goods & Bills
2,GTRX-0000003,GP-006568,SVC-001,2023-11-30 06:00:00,gopay,34229,0.00,Paid,GoPay,False,False,Mobility
3,GTRX-0000004,GP-001920,SVC-001,2023-06-12 14:00:00,gopay,234183,0.00,Paid,GoPay,False,False,Mobility
4,GTRX-0000005,GP-030810,SVC-002,2023-11-01 20:00:00,PayLater,380431,0.00,Paid,PayLater,False,False,Mobility
...,...,...,...,...,...,...,...,...,...,...,...,...
299995,GTRX-0299996,GP-024329,SVC-002,2023-12-26 20:00:00,CASH,327414,0.00,Paid,Cash,False,False,Mobility
299996,GTRX-0299997,GP-008218,SVC-019,2023-02-03 10:00:00,GoPay,17764,0.00,Paid,GoPay,False,False,Offline QRIS
299997,GTRX-0299998,GP-026202,SVC-005,2023-01-29 03:00:00,GoPayLater,253066,0.00,Paid,PayLater,False,False,Food Delivery
299998,GTRX-0299999,GP-000398,SVC-015,2023-06-09 16:00:00,GoPay,502706,0.00,Paid,GoPay,False,False,Offline QRIS


service_category
Digital Goods & Bills    105303
Offline QRIS              89670
Mobility                  44809
Logistics                 30425
Food Delivery             29793
Name: count, dtype: int64

---
### 3.2 Fitur Pilihan (Minimal 2)

Pilih minimal 2 dari: `default_rate_per_user`, `avg_amount_per_service_category`, `is_high_risk_transaction`, `credit_utilization_band`, atau fitur buatan sendiri.

#### ⚙️ Fitur Pilihan 1: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 1


#### ⚙️ Fitur Pilihan 2: [Isi nama fitur]

**✍️ Business value dari fitur ini:**

> 

In [ ]:
# TODO: Implementasi Fitur Pilihan 2


---
## 4. Exploratory Data Analysis

> **Aturan:** Semua analisis menggunakan pandas — tanpa visualisasi.  
> Gunakan `.groupby()`, `.agg()`, `.value_counts()`, filtering, sorting, dan **merge antar tabel** saat dibutuhkan.  
> Setiap jawaban **wajib disertai insight** di markdown cell yang tersedia.

---
### 4.1 Analisis Transaksi & Metode Pembayaran

**Soal 1:** Berapa total GTV (Gross Transaction Value) keseluruhan? Breakdown GTV per `payment_method_clean`. Metode mana yang paling dominan dan apa implikasi bisnisnya?

In [ ]:
# Soal 1


**✍️ Insight:**

> 

**Soal 2:** Berapa distribusi `payment_status` secara keseluruhan? Kemudian breakdown **default rate** per `payment_method_clean`. Apakah GoPayLater memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 2


**✍️ Insight:**

> 

**Soal 3:** Berapa rata-rata, median, dan standar deviasi `amount` per `payment_method_clean`? Apa yang bisa disimpulkan dari perbedaan mean vs median?

In [ ]:
# Soal 3


**✍️ Insight:**

> 

**Soal 4:** Analisis `late_fee`: Berapa persentase transaksi yang dikenakan denda? Berapa total `late_fee` yang terkumpul? Breakdown per `payment_method_clean`.

In [ ]:
# Soal 4


**✍️ Insight:**

> 

---
### 4.2 Analisis Risiko Kredit & Profil User

**Soal 5:** Berapa distribusi `credit_score_tier`? Kemudian bandingkan **default rate** (untuk transaksi GoPayLater) antar `credit_score_tier`. Apakah user dengan credit score rendah memiliki default rate yang lebih tinggi?

In [ ]:
# Soal 5
# Hint: merge df_trx (filter GoPayLater) dengan df_users, lalu groupby credit_score_tier


**✍️ Insight:**

> 

**Soal 6:** Berapa persentase `is_paylater_violation = True`? Breakdown antara: Basic tier pakai PayLater vs Plus tier melebihi limit. Berapa total nilai Rupiah yang terlibat?

In [ ]:
# Soal 6


**✍️ Insight:**

> 

**Soal 7:** Apakah ada korelasi antara `paylater_usage_ratio` dan `is_default`? Bandingkan rata-rata `paylater_usage_ratio` antara transaksi yang Default vs yang tidak.

In [ ]:
# Soal 7


**✍️ Insight:**

> 

**Soal 8:** Berapa distribusi `gopay_tier` di antara user yang pernah Default? Apakah user Basic yang 'membobol' sistem PayLater memiliki default rate lebih tinggi dari user Plus yang sah?

In [ ]:
# Soal 8


**✍️ Insight:**

> 

---
### 4.3 Analisis Layanan & Kategori

**Soal 9:** Berapa total GTV dan jumlah transaksi per `service_category`? Kategori mana yang paling tinggi volumenya?

In [ ]:
# Soal 9


**✍️ Insight:**

> 

**Soal 10:** Berapa **default rate** per `service_category` untuk transaksi GoPayLater? Layanan mana yang paling berisiko untuk dibayar dengan PayLater?

In [ ]:
# Soal 10


**✍️ Insight:**

> 

**Soal 11:** Top 5 `service_name` berdasarkan total `late_fee` yang dikumpulkan. Apakah ini mengindikasikan layanan tertentu lebih sering mengalami keterlambatan pembayaran?

In [ ]:
# Soal 11


**✍️ Insight:**

> 

---
### 4.4 Analisis Sistem & Deteksi Anomali *(Implicit — Business Sense Required)*

> Kamu diminta tim **Risk & Compliance** untuk menyusun laporan investigasi sistem PayLater.  
> Temuan ini akan digunakan untuk: (a) menentukan apakah PayLater perlu di-suspend sementara,  
> (b) mengidentifikasi user yang perlu limit adjustment, dan  
> (c) mengestimasi **total kerugian potensial** dari bug yang ada.

Pilih minimal **2 angle analisis** yang paling relevan untuk menjawab kebutuhan investigasi tersebut.

#### 🔍 Investigasi — Angle 1: [Isi judul]

**✍️ Mengapa kamu memilih angle ini untuk investigasi sistem?**

> 

In [ ]:
# Angle 1


**✍️ Insight & Rekomendasi untuk Tim Risk & Compliance:**

> 

#### 🔍 Investigasi — Angle 2: [Isi judul]

**✍️ Mengapa kamu memilih angle ini?**

> 

In [ ]:
# Angle 2


**✍️ Insight & Rekomendasi:**

> 

---
### 4.5 Credit Risk Profiling *(Implicit — Open Ended)*

> Kamu diminta **Chief Risk Officer GoPay** untuk menyusun rekomendasi perbaikan algoritma credit scoring.  
> Tujuan: menentukan kriteria yang lebih ketat untuk pemberian limit PayLater,  
> sehingga NPL bisa ditekan tanpa terlalu banyak membatasi user yang sebenarnya *creditworthy*.

> 🧠 **Critical Thinking Prompt:**  
> Apakah user dengan credit score rendah **selalu** berisiko?  
> Bagaimana dengan user baru yang belum punya credit score sama sekali?  
> Temukan **sweet spot** antara risk mitigation dan business growth.

**Ekspektasi minimal:**
- Minimal 3 variabel/fitur berbeda yang kamu identifikasi sebagai prediktor default yang signifikan
- Profil 'high-risk user' berdasarkan kombinasi variabel tersebut
- Minimal 1 rekomendasi konkret untuk kebijakan limit PayLater yang berbasis data

**✍️ Definisi 'high-risk user' menurut kamu (dalam konteks kredit GoPay):**

> 

#### 📊 Prediktor Default 1: [Nama Variabel]

In [ ]:
# Prediktor 1


#### 📊 Prediktor Default 2: [Nama Variabel]

In [ ]:
# Prediktor 2


#### 📊 Prediktor Default 3: [Nama Variabel]

In [ ]:
# Prediktor 3


#### 🎯 Profil High-Risk User vs Average User

In [ ]:
# Bandingkan karakteristik high-risk user vs keseluruhan user PayLater


**✍️ Rekomendasi Kebijakan Limit PayLater untuk Chief Risk Officer:**

> 

---
## 5. Export Clean Dataset

In [ ]:
# Gabungkan ketiga tabel menjadi satu dataframe final
# Gunakan LEFT JOIN dengan df_trx sebagai tabel utama
# Sertakan semua fitur baru yang telah dibuat

# TODO: Implementasi JOIN
# df_final = df_trx.merge(df_users[...], on='user_id', how='left')
#                  .merge(df_services[...], on='service_id', how='left')

# Export
# df_final.to_csv('gopay_clean.csv', index=False)
# print(f'Dataset berhasil disimpan: gopay_clean.csv')
# print(f'Shape final: {df_final.shape}')
# print(f'Kolom baru yang ditambahkan: {[c for c in df_final.columns if c not in df_trx_raw.columns]}')


---
## 6. Ringkasan & Refleksi

**Keputusan Data Cleaning yang paling challenging dan mengapa:**

> 

**Temuan paling menarik dari EDA (khususnya terkait risiko kredit):**

> 

**Rekomendasi bisnis utama yang bisa diberikan kepada tim Risk Management GoPay:**

> 